In [1]:
import torch
import torch.nn as nn

In [ ]:
class ConvBN(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self,x):
        return self.conv(x)

In [ ]:
class ConvBNReLU(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.conv = nn.Sequential(
            ConvBN(in_channels, out_channels, kernel_size, stride, padding),
            nn.ReLU(),
        )

    def forward(self,x):
        return self.conv(x)

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.relu = nn.ReLU()
        self.shortcut = nn.Identity()
        if in_channels != out_channels or stride != 1:
            self.shortcut = ConvBN(in_channels, out_channels, kernel_size=1, stride=stride, padding=0)
        self.conv = nn.Sequential(
            ConvBNReLU(in_channels, out_channels, 3, stride=stride, padding=1),
            ConvBN(out_channels, out_channels, 3, stride=1, padding=1),
        )

    def forward(self,x):
        return self.relu(self.conv(x) + self.shortcut(x))

In [ ]:
class ResNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = ConvBNReLU(1,64,7,2,3)
        self.block1 = ResidualBlock(64,64)
        self.block2 = ResidualBlock(64,64)
        self.block3 = ResidualBlock(64,128,2)
        self.block4 = ResidualBlock(128,128)
        self.block5 = ResidualBlock(128,256,2)
        self.block6 = ResidualBlock(256,256)
        self.block7 = ResidualBlock(256,512,2)
        self.block8 = ResidualBlock(512,512)
        self.fc1 = nn.Linear(in_features=512, out_features=10)
        self.max_pool = nn.MaxPool2d(kernel_size=(3,3), stride=(2,2), padding=1)
        self.average_pool = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x):
        x = self.conv1(x)
        x = self.max_pool(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.block5(x)
        x = self.block6(x)
        x = self.block7(x)
        x = self.block8(x)
        x = self.average_pool(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        return x